## 11. Rigorous Validation of the Operator Commutator & its Applications

In classical mechanics, the Poisson bracket $\[a, b\]$ measures the failure of two observables to commute. In quantum mechanics and microlocal analysis, this role is played by the operator commutator:
$$
[A, B] = AB - BA
$$
where $A$ and $B$ are pseudo-differential operators ($\Psi$DOs).

According to the Weyl/Kohn-Nirenberg calculus, the symbol of the commutator $[A, B]$ has an asymptotic expansion whose leading-order term is proportional to the Poisson bracket of their symbols:
$$
\sigma([A, B]) = -i \{a, b\} + \mathcal{O}(\xi^{m+k-2})
$$
where $\{a, b\} = \partial_\xi a \partial_x b - \partial_x a \partial_y b$.

Let's rigorously test this identity using `psiop`:
1. **Symbolic Verification:** We will compute the commutator of a spatially varying potential/speed operator and a momentum operator to see the exact cancellation of higher-order terms.
2. **Physical Application (Conservation Laws / Egorov's Theorem):** We will show how the commutator determines whether an operator commutes with a Hamiltonian, governing the quantum/microlocal transport of energy.

In [ ]:
from psiop import PseudoDifferentialOperator
import sympy as sp
from sympy import symbols, Function, diff, simplify, I

x, xi = symbols('x xi', real=True)

### Test A: The Commutator of Position and Momentum (The Canonical Commutator)

Let's start with the most fundamental commutator in physics: $[x, D_x]$ where $D_x = -i \partial_x$.
In pseudo-differential calculus, the position operator $A = x$ has symbol $a(x,\xi) = x$, and the momentum operator $B = D_x$ has symbol $b(x,\xi) = \xi$.

Since $[x, D_x] = i \mathbb{I}$, the symbol of the commutator should be exactly $i$ (which is $-i \{x, \xi\} = -i(0 - 1) = i$).

In [ ]:
# 1. Define Position Operator A = x
A_sym = x
A_op = PseudoDifferentialOperator(A_sym, [x], mode='symbol')

# 2. Define Momentum Operator B = xi
B_sym = xi
B_op = PseudoDifferentialOperator(B_sym, [x], mode='symbol')

# 3. Compute Compositions: AB and BA
AB_sym = A_op.compose_asymptotic(B_op, order=2, mode='kn')
BA_sym = B_op.compose_asymptotic(A_op, order=2, mode='kn')

# 4. Calculate the Commutator symbol: [A, B] = AB - BA
comm_sym = simplify(AB_sym - BA_sym)

print("Symbol of AB:")
sp.pprint(simplify(AB_sym))

print("\nSymbol of BA:")
sp.pprint(simplify(BA_sym))

print("\nSymbol of the Commutator [x, D_x]:")
sp.pprint(comm_sym)

print("\nIs the commutator exactly equal to I (imaginary unit)?", sp.nsimplify(comm_sym - I)==0)

### Test B: Variable Speed and Kinetic Energy (The Microlocal Commutator)

Let's move to a non-trivial setting: a variable wave speed coefficient $c(x)$ and a kinetic energy operator.
Let $A = c(x)$ (multiplication operator, symbol $a = c(x)$) and let $B = -\partial_x^2$ (Laplacian, symbol $b = \xi^2$).

Because the spatial coefficient does not commute with derivatives, the commutator $[c(x), -\partial_x^2]$ will yield a first-order differential operator.
The Poisson bracket is:
$$
\{a, b\} = \partial_\xi a \partial_x b - \partial_x a \partial_\xi b = 0 - c'(x)(2\xi) = -2c'(x)\xi
$$
Thus, we expect the leading order symbol of the commutator to be:
$$
\sigma([A, B]) \approx -i \{a, b\} = 2 i c'(x) \xi
$$
Let's verify this using asymptotic composition up to order 2.

In [ ]:
# Define spatially varying wave speed c(x)
c = Function('c')(x)

# 1. Operator A = c(x)
A_var_sym = c
A_var_op = PseudoDifferentialOperator(A_var_sym, [x], mode='symbol')

# 2. Operator B = -d^2/dx^2 -> symbol xi^2
B_var_sym = xi**2
B_var_op = PseudoDifferentialOperator(B_var_sym, [x], mode='symbol')

# 3. Compute both compositions asymptotically up to order 2
AB_var_sym = A_var_op.compose_asymptotic(B_var_op, order=2, mode='kn')
BA_var_sym = B_var_op.compose_asymptotic(A_var_op, order=2, mode='kn')

# 4. [A, B] = AB - BA
comm_var_sym = simplify(AB_var_sym - BA_var_sym)

print("--- MICROLOCAL COMMUTATOR ---")
print("Symbol of A = c(x):")
sp.pprint(A_var_sym)

print("\nSymbol of B = -d^2/dx^2:")
sp.pprint(B_var_sym)

print("\nSymbol of [c(x), -\\Delta]:")
sp.pprint(comm_var_sym)

# Verify Poisson bracket relationship
poisson_bracket = -I * (diff(A_var_sym, xi)*diff(B_var_sym, x) - diff(A_var_sym, x)*diff(B_var_sym, xi))
print("\nExpected leading-order term (-i * {a, b}):")
sp.pprint(poisson_bracket)

### Analyzing the Commutator Error Term

Let's inspect the remaining terms of the commutator. Since the full asymptotic expansion of the Kohn-Nirenberg composition contains higher-order derivatives, the commutator $[A, B]$ will contain sub-leading terms. 

Let's extract the coefficients of the commutator by power of $\xi$ to see how the Poisson bracket acts as the dominant high-frequency driver, while a lower-order quantum correction sits at $\mathcal{O}(\xi^0)$.

In [ ]:
comm_expanded = sp.expand(comm_var_sym)

print("Coefficients of the Commutator [c(x), -\\Delta] by power of ξ:")
coeffs = {}
for n in range(2, -2, -1):
    c_n = sp.simplify(comm_expanded.coeff(xi, n))
    if c_n != 0:
        coeffs[n] = c_n
        print(f"\n  O(ξ^{n}):")
        sp.pprint(c_n)

leading_term = coeffs.get(1, 0) * xi
correction_term = coeffs.get(0, 0)

print("\n💡 Mathematical Insight:")
print("The O(ξ¹) term matches the expected Poisson bracket term: 2*i*c'(x)*ξ.")
print("The sub-leading O(ξ⁰) term is:")
sp.pprint(correction_term)
print("This represents the second-order derivative contribution c''(x), which completes")
print("the exact operator identity: [c, -d^2/dx^2] = 2 c'(x) d/dx + c''(x).")

### Conclusion: Commutators, Energy Conservation, and Egorov's Theorem

This notebook demonstrated the power of `psiop`'s symbolic pipeline to resolve the algebraic structures underlying quantum and wave physics:

1. **Poisson Bracket Alignment:** The leading-order symbol of a commutator $[A, B]$ is equivalent to $-i\{a,b\}$. In phase space, this links quantum/pseudo-differential commutativity directly to classical symplectic geometry.

2. **Quantum/Microlocal Correction:** Beyond the leading-order term, lower-order symbol components (such as the $\mathcal{O}(\xi^0)$ term $c''(x)$) naturally emerge. These represent higher-order spatial derivatives that are often omitted in simplified ray-tracing calculations but are essential for exact operator representations.

3. **Physical Significance (Egorov's Theorem):** If $H$ is a Hamiltonian operator (e.g., $H = -\Delta + V(x)$), the time evolution of any observable operator $A(t) = e^{itH} A e^{-itH}$ satisfies Heisenberg's equation of motion:
$$
\frac{d}{dt} A(t) = i [H, A(t)]
$$
At the symbol level, this translates to $\partial_t a = \{h, a\} + \mathcal{O}(\xi^{m-1})$, which dictates that energy and wave amplitudes are transported along classical rays (geodesics). `psiop` allows us to verify these conservation structures symbolic step by symbolic step.